In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 1. Cargar los datos cuartohorarios
# Asegúrate de que el índice es datetime y tiene una frecuencia fija
df = pd.read_csv('../data/raw/precios_historicos.csv', index_col='datetime', parse_dates=True)

# Llenar posibles huecos vacíos (NA) interpolando linealmente
df['precio_mwh'] = df['precio_mwh'].interpolate()

In [ ]:
# 2. Descomposición de la serie (Estacionalidad s=96 para cuartohorario)
# Usamos un par de semanas de datos para que el gráfico sea legible
df_sample = df.iloc[:96 * 14] 
descomposicion = seasonal_decompose(df_sample['precio_mwh'], model='additive', period=96)

fig = descomposicion.plot()
fig.set_size_inches(12, 8)
plt.suptitle('Descomposición Aditiva: Precio Mercado Cuartohorario', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Forzar la Estacionariedad
# Paso A: Aplicar logaritmo para estabilizar la varianza
# Nota: Sumamos una constante por si hay precios a 0 o negativos (común en mercado eléctrico)
constante = abs(df['precio_mwh'].min()) + 1
df['precio_log'] = np.log(df['precio_mwh'] + constante)

# Paso B: Diferenciación para estabilizar la media (Xd_t = X_t - X_t-1)
df['precio_diff'] = df['precio_log'].diff().dropna()

In [ ]:
# 4. Análisis de Autocorrelación (ACF) y Autocorrelación Parcial (PACF)
# Esto nos dirá qué parámetros (p, q) necesitará nuestro futuro modelo
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# ACF nos ayuda a ver la correlación de la señal con copias retrasadas de sí misma
plot_acf(df['precio_diff'].dropna(), lags=100, ax=axes[0], title="ACF (Diferenciada)")
# PACF elimina el efecto de los valores intermedios para ver la relación directa
plot_pacf(df['precio_diff'].dropna(), lags=100, ax=axes[1], title="PACF (Diferenciada)")

plt.show()